In [3]:
from config import API_KEY, get_connection

with get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT last_run_timestamp FROM raw.pipeline_metadata WHERE pipeline_name = 'tmdb_bulk_insert'
        """)
        last_run = cur.fetchone()[0]
        print(last_run)

2026-05-15 00:00:00


In [4]:
import datetime
import requests
import psycopg2
from psycopg2.extras import Json

all_ids = []
current_page = 1

while True:
    today = datetime.date.today()
    url = f"https://api.themoviedb.org/3/movie/changes"
    params = {
        'api_key': API_KEY,
        'start_date': last_run.strftime('%Y-%m-%d'),
        'end_date': today.strftime('%Y-%m-%d'),
        'page': current_page
    }

    response = requests.get(url, params=params).json()

    page_ids = [m['id'] for m in response.get('results', [])]
    all_ids.extend(page_ids)

    if current_page >= response.get('total_pages'):
        break

    current_page += 1

print(all_ids)

[1695125, 1473148, 1695130, 1687025, 1694942, 1694733, 1694914, 1694806, 1493269, 1613874, 1659077, 1667595, 1670416, 1673947, 1673953, 1677479, 1684211, 1685907, 1687008, 1688861, 1691860, 1692808, 1694374, 454639, 1061821, 1228710, 1003494, 1182935, 1401459, 1404582, 1469342, 1548931, 1550622, 1573802, 1655419, 1665750, 1666146, 1668922, 1673938, 1675735, 1694934, 1368314, 1694508, 1694748, 1694787, 1694966, 1666823, 1688207, 1694487, 1694832, 1694851, 1694866, 1694958, 1691418, 1694564, 1675086, 1694433, 1694792, 1694860, 1695076, 1695059, 1660550, 1694761, 1692713, 1654102, 1657236, 1528885, 1657322, 1694504, 1694378, 1694849, 1694947, 1694426, 1694728, 1694409, 1694462, 1694282, 1602431, 1489678, 1341684, 96343, 1110034, 1632181, 393069, 1695131, 31861, 1233243, 1691839, 1641045, 1695129, 8047, 1279379, 1695132, 1118403, 182496, 1643067, 675356, 1339713, 345690, 1695124, 1480503, 56937, 1693346, 1641030, 1666712, 1207655, 1630401, 1641026, 1694425, 1472277, 1693705, 1695103, 88224

In [5]:
len(all_ids)

18824